# Amazon ML — submit the evaluated embedding model

This notebook produces predictions for **all test Source 1 businesses** using the matcher from your completed full-pool embedding experiment (local macro F0.5 ~0.8864).
It does not retrain, increase the training sample, or change the cutoff/search settings.

**Attach two inputs to a new Kaggle notebook:**
1. The original challenge dataset, including `student_resource/utils/validate_submission.py`.
2. The **saved output of your completed full-pool embedding notebook**. It must contain `matcher.pkl` and `experiment_results.json`. These are the model you already trained and its metadata. The small results ZIP can also supply these files if extracted into an attached input.

Enable **GPU** and **Internet**. Use **Save Version → Save & Run All** so completed outputs are retained. Keep the previous experiment saved separately.

The notebook uses only the saved matcher and metadata from the old run, without copying its large training caches. It creates new embeddings/indexes for test S2/S3, because those records differ from the training pool.
Allow several hours; prediction logs estimate remaining time after the first batch. Start with at least ~13 GiB free working disk. Nothing is uploaded to the challenge portal automatically.


In [ ]:
%pip install -q "numpy==2.0.2" "pandas==2.3.3" "scikit-learn==1.6.1" "faiss-cpu==1.11.0" "transformers==5.0.0" safetensors sentencepiece


In [ ]:
from pathlib import Path
import os,sys,shutil
DATA=Path('/kaggle/input/datasets/spoortimorabad/ml-challenge-dataset/student_resource/dataset')
WORK=Path('/kaggle/working/embedding_submission_v1')
# Leave blank to find the previous completed experiment automatically.
# If more than one is attached, paste the folder containing its matcher.pkl here.
MODEL_DIR=''
BATCH_SIZE=64
WORK.mkdir(parents=True,exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM']='false'
import torch
assert torch.cuda.is_available(),'Enable a GPU before running this notebook.'
assert (DATA/'test/test_source1.tsv').exists(),'Update DATA to your challenge dataset path.'
assert (DATA.parent/'utils/validate_submission.py').exists(),'Attach the original challenge utils folder too.'
print('GPU:',torch.cuda.get_device_name(0))
print('Free working disk GiB:',round(shutil.disk_usage(WORK).free/2**30,2))


## Write and check the submission pipeline
The code is embedded here, so the notebook is self-contained apart from the dataset and your trained matcher. Existing caches require the same inputs and code; use a new WORK folder if changing the experiment.

In [ ]:
CORE_SOURCE='import os, json, time, gc, sqlite3, hashlib, shutil, pickle, re, unicodedata\nfrom pathlib import Path\nfrom functools import lru_cache\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom threadpoolctl import threadpool_limits\nCOLS = [\'entity_id\', \'business_name\', \'business_address\', \'country\']\n\nLEGAL = set(\'inc incorporated llc llp ltd limited pvt private corp corporation company co plc sarl sas eurl\'.split())\n\nADDR_ALIAS = {\'road\': \'rd\', \'street\': \'st\', \'avenue\': \'ave\', \'drive\': \'dr\', \'lane\': \'ln\', \'boulevard\': \'blvd\', \'apartment\': \'apt\', \'suite\': \'ste\'}\n\nFEATURES = [\'name_exact\', \'core_name_exact\', \'name_trigram_jaccard\', \'name_token_jaccard\', \'name_token_containment\', \'name_length_ratio\', \'address_exact\', \'address_trigram_jaccard\', \'address_token_jaccard\', \'address_token_containment\', \'address_missing\', \'number_jaccard\', \'number_disjoint\', \'unit_equal\', \'unit_conflict\', \'country_equal\', \'name_token_count\', \'candidate_name_token_count\']\n\ndef log(s):\n    print(s, flush=True)\n\ndef norm(s):\n    s = unicodedata.normalize(\'NFKC\', str(s)).casefold()\n    return \' \'.join(re.findall(\'[^\\\\W_]+\', s))\n\ndef canonical_address(s):\n    return \' \'.join((ADDR_ALIAS.get(t, t) for t in norm(s).split() if t not in {\'null\', \'none\'}))\n\ndef grams(s):\n    return frozenset((s[i:i + 3] for i in range(max(0, len(s) - 2))))\n\ndef unit_id(s):\n    m = re.search(\'\\\\b(?:unit|suite|ste|apt|apartment|flat)\\\\s*(?:no\\\\.?\\\\s*)?[:#-]?\\\\s*([\\\\w]+)\', unicodedata.normalize(\'NFKC\', s).casefold())\n    return m.group(1) if m else \'\'\n\n@lru_cache(maxsize=30000)\ndef prepared(name, address, country):\n    n = norm(name)\n    a = canonical_address(address)\n    nt, at = (frozenset(n.split()), frozenset(a.split()))\n    core = tuple(sorted((t for t in nt if t not in LEGAL)))\n    nums = frozenset(re.findall(\'\\\\d+[a-z]*\', a))\n    return (n, a, nt, at, core, nums, grams(n), grams(a), unit_id(address), norm(country))\n\ndef jaccard(a, b):\n    return len(a & b) / len(a | b) if a or b else 0.0\n\ndef containment(a, b):\n    return len(a & b) / min(len(a), len(b)) if a and b else 0.0\n\ndef features(left, right):\n    n, a, nt, at, c, nums, ng, ag, u, country = prepared(*left[1:])\n    m, b, mt, bt, d, other_nums, mg, bg, v, other_country = prepared(*right[1:])\n    return [float(bool(n) and n == m), float(bool(c) and c == d), jaccard(ng, mg), jaccard(nt, mt), containment(nt, mt), min(len(n), len(m)) / max(len(n), len(m), 1), float(bool(a) and a == b), jaccard(ag, bg), jaccard(at, bt), containment(at, bt), float(not a or not b), jaccard(nums, other_nums), float(bool(nums) and bool(other_nums) and (not nums & other_nums)), float(bool(u) and bool(v) and (u == v)), float(bool(u) and bool(v) and (u != v)), float(bool(country) and country == other_country), len(nt), len(mt)]\n\ndef read_chunks(path, cols=COLS):\n    return pd.read_csv(path, sep=\'\\t\', dtype=str, keep_default_na=False, usecols=cols, chunksize=100000)\n\ndef reproduce_reference_sample(data):\n    from sklearn.model_selection import train_test_split\n    rng = np.random.default_rng(2026)\n    sample = pd.DataFrame()\n    for chunk in read_chunks(data / \'train/train_source1.tsv\'):\n        chunk = chunk[COLS].copy()\n        chunk[\'_priority\'] = rng.random(len(chunk))\n        sample = pd.concat([sample, chunk], ignore_index=True).nsmallest(4000, \'_priority\')\n    sample = sample.drop(columns=\'_priority\').reset_index(drop=True)\n    assert len(sample) == 4000 and sample.entity_id.is_unique\n    ids = set(sample.entity_id)\n    truth = {}\n    for chunk in read_chunks(data / \'train/train_ground_truth.tsv\', [\'source1_entity_id\', \'matched_entity_ids\']):\n        for row in chunk[chunk.source1_entity_id.isin(ids)].itertuples(index=False):\n            truth[row.source1_entity_id] = set((x.strip() for x in row.matched_entity_ids.split(\',\') if x.strip()))\n    assert set(truth) == ids\n    strata = sample.country + \'|\' + sample.entity_id.map(lambda x: str(not truth[x]))\n    train_idx, held = train_test_split(np.arange(4000), test_size=0.25, random_state=2026, stratify=strata)\n    tune_idx, eval_idx = train_test_split(held, test_size=0.5, random_state=2027, stratify=strata.iloc[held])\n    sample[\'split\'] = \'train\'\n    sample.loc[tune_idx, \'split\'] = \'tune\'\n    sample.loc[eval_idx, \'split\'] = \'evaluation\'\n    return (sample, truth)\n\ndef clean_embedding_text(s):\n    return \' \'.join(unicodedata.normalize(\'NFKC\', str(s)).split())\n\ndef text_view(frame, view):\n    if view == \'name\':\n        return [\'query: \' + clean_embedding_text(x) for x in frame.business_name]\n    return [\'query: business name: \' + clean_embedding_text(n) + \'; address: \' + clean_embedding_text(a) for n, a in zip(frame.business_name, frame.business_address)]\n\nMODEL_ID=\'intfloat/multilingual-e5-small\'\nREVISION=\'614241f622f53c4eeff9890bdc4f31cfecc418b3\'\nDIM=384\nVERSION=\'full-pool-e5-v1\'\n\ndef atomic_json(path, value):\n    tmp=Path(str(path)+\'.tmp\')\n    tmp.write_text(json.dumps(value,indent=2))\n    os.replace(tmp,path)\n\ndef initialize(data,work):\n    work=Path(work); data=Path(data); work.mkdir(parents=True,exist_ok=True)\n    files=[data/f\'train/train_source{i}.tsv\' for i in (1,2,3)]+[data/\'train/train_ground_truth.tsv\']\n    for p in files:\n        assert p.exists(),f\'Missing dataset: {p}\'\n    config={\'version\':VERSION,\'model\':MODEL_ID,\'revision\':REVISION,\'dim\':DIM,\n            \'max_length\':256,\'text\':\'query: business name: NAME; address: ADDRESS\',\n            \'files\':[(str(p),p.stat().st_size) for p in files],\n            \'nlist_max\':2048,\'pq_m\':48,\'pq_bits\':8,\'nprobe\':64,\'overfetch\':200,\n            \'candidate_k\':20,\'sample_seed\':2026,\'sample_size\':4000}\n    cp=work/\'configuration.json\'\n    if cp.exists():\n        assert json.loads(cp.read_text())==json.loads(json.dumps(config)), \'Configuration changed. Use a new WORK folder.\'\n    else: atomic_json(cp,config)\n    log(f\'Free disk: {shutil.disk_usage(work).free/2**30:.1f} GiB\')\n    return data,work\n\nclass Encoder:\n    def __init__(self,batch_size=64):\n        import torch\n        from transformers import AutoTokenizer,AutoModel\n        assert torch.cuda.is_available(),\'Enable a GPU in Kaggle Settings, then restart/run all.\'\n        self.torch=torch; self.batch_size=batch_size\n        self.tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False)\n        self.model=AutoModel.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False,\n                                           use_safetensors=True).cuda().half().eval()\n        log(\'GPU: \'+torch.cuda.get_device_name(0))\n    def encode(self,frame):\n        texts=text_view(frame,\'combined\'); parts=[]; torch=self.torch\n        with torch.inference_mode():\n            for start in range(0,len(texts),self.batch_size):\n                inp=self.tokenizer(texts[start:start+self.batch_size],max_length=256,\n                                   padding=True,truncation=True,return_tensors=\'pt\').to(\'cuda\')\n                hidden=self.model(**inp).last_hidden_state.float()\n                mask=inp[\'attention_mask\'].unsqueeze(-1)\n                vec=(hidden*mask).sum(1)/mask.sum(1).clamp(min=1)\n                vec=torch.nn.functional.normalize(vec,p=2,dim=1)\n                parts.append(vec.cpu().numpy())\n        return np.concatenate(parts).astype(\'float32\') if parts else np.empty((0,DIM),np.float32)\n\ndef open_records(work):\n    db=sqlite3.connect(work/\'records.sqlite\')\n    db.execute(\'PRAGMA cache_size=-65536\')\n    db.execute(\'CREATE TABLE IF NOT EXISTS records (rid INTEGER PRIMARY KEY, entity_id TEXT, business_name TEXT, business_address TEXT, country TEXT)\')\n    db.execute(\'CREATE TABLE IF NOT EXISTS progress (source INTEGER PRIMARY KEY, done INTEGER, complete INTEGER)\')\n    db.commit()\n    return db\n\ndef encode_pool(data,work,encoder,split="train"):\n    """Vectors flushed before SQLite transaction commits; uncommitted rows overwritten on restart."""\n    manifest_path=work/\'pool_manifest.json\'\n    if not manifest_path.exists():\n        counts={}\n        for source in (2,3):\n            counts[str(source)]=sum(len(c) for c in read_chunks(data/f\'{split}/{split}_source{source}.tsv\',[\'entity_id\']))\n        atomic_json(manifest_path,{\'counts\':counts,\'total\':sum(counts.values())})\n    manifest=json.loads(manifest_path.read_text()); total=manifest[\'total\']\n    vp=work/\'vectors.f16\'; needed=total*DIM*2\n    db=open_records(work)\n    exists=vp.exists()\n    if not exists:\n        assert db.execute(\'SELECT COUNT(*) FROM records\').fetchone()[0]==0,\'Vectors missing but database has progress. Restore both or use new WORK.\'\n        # Includes space for metadata DB, compressed indexes, model cache and reserve.\n        assert shutil.disk_usage(work).free>needed+5*2**30, \'Need about 13 GiB free before full-pool encoding. Use a fresh notebook.\'\n    else: assert vp.stat().st_size==needed,\'Incomplete vector allocation. Use new WORK directory.\'\n    vectors=np.memmap(vp,mode=\'r+\' if exists else \'w+\',dtype=np.float16,shape=(total,DIM))\n    base=0; started=time.time(); encoded=0\n    for source in (2,3):\n        saved=db.execute(\'SELECT done,complete FROM progress WHERE source=?\',(source,)).fetchone()\n        done=saved[0] if saved else 0\n        if saved and saved[1]:\n            log(f\'Source {source}: already encoded {done:,} rows\'); base+=manifest[\'counts\'][str(source)]; continue\n        log(f\'Source {source}: resume at {done:,}; reading input from beginning to that position\')\n        seen=0\n        chunks=pd.read_csv(data/f\'{split}/{split}_source{source}.tsv\',sep=\'\\t\',dtype=str,\n                           keep_default_na=False,usecols=COLS,chunksize=10000)\n        for frame in chunks:\n            end=seen+len(frame)\n            if end<=done: seen=end; continue\n            frame=frame.iloc[max(0,done-seen):][COLS]\n            first=max(done,seen); batch=encoder.encode(frame)\n            assert batch.shape==(len(frame),DIM) and np.isfinite(batch).all()\n            vectors[base+first:base+end]=batch.astype(np.float16); vectors.flush()\n            rows=[(base+first+j,str(r.entity_id),str(r.business_name),str(r.business_address),norm(r.country))\n                  for j,r in enumerate(frame.itertuples(index=False))]\n            with db:\n                db.executemany(\'INSERT INTO records VALUES (?,?,?,?,?)\',rows)\n                db.execute(\'INSERT OR REPLACE INTO progress VALUES (?,?,0)\',(source,end))\n            encoded+=len(frame); seen=end; done=end\n            if end%100000==0:\n                rate=encoded/max(time.time()-started,1)\n                remaining=total-(base+end)\n                log(f\'Source {source}: {end:,}; {rate:.0f} rows/sec; remaining encoding ~{remaining/rate/60:.1f} min\')\n                assert shutil.disk_usage(work).free>1.5*2**30,\'Low disk. Progress saved; free space before resuming.\'\n        assert done==manifest[\'counts\'][str(source)]\n        with db: db.execute(\'INSERT OR REPLACE INTO progress VALUES (?,?,1)\',(source,done))\n        base+=done\n    assert db.execute(\'SELECT COUNT(*) FROM records\').fetchone()[0]==total\n    atomic_json(work/\'encoding_complete.json\',{\'total\':total})\n    db.close(); del vectors\n    log(f\'Encoding complete: {total:,} target records.\')\n\ndef build_indexes(work):\n    """IVF-PQ index per country. Completed countries and trained quantizers are reusable."""\n    import faiss\n    faiss.omp_set_num_threads(min(4,os.cpu_count() or 2))\n    total=json.loads((work/\'encoding_complete.json\').read_text())[\'total\']\n    vectors=np.memmap(work/\'vectors.f16\',mode=\'r\',dtype=np.float16,shape=(total,DIM))\n    db=open_records(work)\n    countries=[r[0] for r in db.execute(\'SELECT DISTINCT country FROM records\')]\n    catalog=[]\n    for country in sorted(countries):\n        token=hashlib.sha256(country.encode()).hexdigest()[:12]\n        path=work/f\'index_{token}.faiss\'; trained=work/f\'trained_{token}.faiss\'\n        count=db.execute(\'SELECT COUNT(*) FROM records WHERE country=?\',(country,)).fetchone()[0]\n        catalog.append({\'country\':country,\'file\':path.name,\'count\':count})\n        if path.exists():\n            check=faiss.read_index(str(path)); assert check.ntotal==count\n            del check; log(f\'{country}: completed index reused\'); continue\n        ids=np.fromiter((r[0] for r in db.execute(\'SELECT rid FROM records WHERE country=? ORDER BY rid\',(country,))),dtype=np.int64,count=count)\n        # At least 50 training vectors per IVF centroid, up to 100k training vectors.\n        nlist=min(2048,max(1,count//50))\n        if trained.exists(): index=faiss.read_index(str(trained))\n        elif count<10000:\n            index=faiss.IndexIDMap2(faiss.IndexFlatIP(DIM))\n        else:\n            index=faiss.IndexIVFPQ(faiss.IndexFlatIP(DIM),DIM,nlist,48,8,faiss.METRIC_INNER_PRODUCT)\n            index.cp.seed=2026; index.pq.cp.seed=2026\n            train_ids=np.random.default_rng(2026).choice(ids,size=min(count,110000),replace=False)\n            train_vectors=np.asarray(vectors[np.sort(train_ids)],dtype=np.float32)\n            faiss.normalize_L2(train_vectors)\n            log(f\'{country}: training IVF-PQ on {len(train_vectors):,} vectors; {nlist} cells\')\n            index.train(train_vectors); del train_vectors\n            faiss.write_index(index,str(trained)+\'.tmp\'); os.replace(str(trained)+\'.tmp\',trained)\n        started=time.time()\n        # An interrupted country is re-added from its saved trained quantizer.\n        for start in range(0,count,25000):\n            batch_ids=ids[start:start+25000]\n            x=np.asarray(vectors[batch_ids],dtype=np.float32); faiss.normalize_L2(x)\n            index.add_with_ids(x,batch_ids)\n            if start%250000==0: log(f\'{country}: indexed {min(start+25000,count):,}/{count:,}; {(time.time()-started)/60:.1f} min\')\n        assert index.ntotal==count\n        faiss.write_index(index,str(path)+\'.tmp\'); os.replace(str(path)+\'.tmp\',path)\n        del index,ids; gc.collect()\n    atomic_json(work/\'index_catalog.json\',catalog)\n    db.close(); del vectors\n    log(\'All country indexes ready.\')\n\ndef prepare_queries(data,work,encoder):\n    sample_path=work/\'sample.pkl\'\n    if sample_path.exists():\n        with sample_path.open(\'rb\') as f: sample,truth=pickle.load(f)\n    else:\n        sample,truth=reproduce_reference_sample(data)\n        with Path(str(sample_path)+\'.tmp\').open(\'wb\') as f: pickle.dump((sample,truth),f)\n        os.replace(str(sample_path)+\'.tmp\',sample_path)\n        sample.to_csv(work/\'sample.tsv\',sep=\'\\t\',index=False)\n    path=work/\'query_vectors.npy\'\n    if path.exists(): q=np.load(path)\n    else:\n        q=encoder.encode(sample)\n        with Path(str(path)+\'.tmp\').open(\'wb\') as f: np.save(f,q)\n        os.replace(str(path)+\'.tmp\',path)\n    assert q.shape==(len(sample),DIM)\n    log(f\'Sample splits: {sample.split.value_counts().to_dict()}\')\n    return sample,truth,q\n\ndef retrieve(work,sample,q,nprobe=64,overfetch=200,save_k=50):\n    import faiss\n    query_key=hashlib.sha256((\'|\'.join(sample.entity_id)).encode()).hexdigest()[:12]\n    cache=work/f\'retrieval_{query_key}_p{nprobe}_r{overfetch}_k{save_k}.npz\'\n    if cache.exists():\n        saved=np.load(cache); return saved[\'ids\'],saved[\'scores\']\n    total=json.loads((work/\'encoding_complete.json\').read_text())[\'total\']\n    vectors=np.memmap(work/\'vectors.f16\',mode=\'r\',dtype=np.float16,shape=(total,DIM))\n    ids=np.full((len(q),save_k),-1,np.int64); scores=np.full((len(q),save_k),-1,np.float32)\n    countries=sample.country.map(norm).to_numpy()\n    for item in json.loads((work/\'index_catalog.json\').read_text()):\n        qi=np.flatnonzero(countries==item[\'country\'])\n        if not len(qi): continue\n        index=faiss.read_index(str(work/item[\'file\']))\n        if hasattr(index,\'nprobe\'): index.nprobe=min(nprobe,index.nlist)\n        log(f"Searching {len(qi)} businesses in {item[\'country\']}: {index.ntotal:,} records")\n        for start in range(0,len(qi),64):\n            batch=qi[start:start+64]\n            _,approx=index.search(np.ascontiguousarray(q[batch],dtype=np.float32),overfetch)\n            for i,neighbors in zip(batch,approx):\n                neighbors=np.unique(neighbors[neighbors>=0])\n                if not len(neighbors): continue\n                x=np.asarray(vectors[neighbors],dtype=np.float32)\n                x/=np.maximum(np.linalg.norm(x,axis=1,keepdims=True),1e-12)\n                exact=x@q[i]\n                order=np.lexsort((neighbors,-exact))[:save_k]\n                chosen=neighbors[order]; k=len(chosen)\n                ids[i,:k]=chosen; scores[i,:k]=exact[order]\n        del index; gc.collect()\n    with Path(str(cache)+\'.tmp\').open(\'wb\') as f: np.savez_compressed(f,ids=ids,scores=scores)\n    os.replace(str(cache)+\'.tmp\',cache)\n    return ids,scores\n\ndef fetch_records(work,ids):\n    db=open_records(work); rows={}\n    valid=np.unique(ids[ids>=0])\n    for start in range(0,len(valid),800):\n        batch=[int(x) for x in valid[start:start+800]]\n        for r in db.execute(\'SELECT * FROM records WHERE rid IN (\'+\',\'.join(\'?\'*len(batch))+\')\',batch):\n            rows[r[0]]=tuple(r[1:])\n    db.close()\n    assert len(rows)==len(valid)\n    return rows\n\ndef retrieval_report(sample,truth,ids,records):\n    out=[]\n    for split in (\'train\',\'tune\',\'evaluation\'):\n        indices=np.flatnonzero(sample.split.to_numpy()==split)\n        if not len(indices): continue\n        for k in (10,20,50):\n            found=total=0; sizes=[]\n            for i in indices:\n                gt=truth[sample.iloc[i].entity_id]\n                pred={records[int(r)][0] for r in ids[i,:k] if r>=0}\n                found+=len(gt&pred); total+=len(gt); sizes.append(len(pred))\n            out.append(dict(split=split,k=k,found=found,true_links=total,\n                            recall=found/total if total else 1,mean_candidates=float(np.mean(sizes)),\n                            p95_candidates=float(np.percentile(sizes,95)),max_candidates=max(sizes)))\n    return pd.DataFrame(out)\n\nMATCH_FEATURES=FEATURES+[\'embedding_cosine\']\ndef make_pairs(sample,truth,ids,scores,records,k=20):\n    x=[]; y=[]; owners=[]; candidate_ids=[]\n    for i,left in enumerate(sample[COLS].itertuples(index=False,name=None)):\n        for rank,rid in enumerate(ids[i,:k]):\n            if rid<0: continue\n            right=records[int(rid)]\n            x.append(features(left,right)+[float(scores[i,rank])])\n            y.append(int(right[0] in truth[left[0]])); owners.append(i); candidate_ids.append(right[0])\n    return np.asarray(x,np.float32),np.asarray(y,np.uint8),np.asarray(owners,np.int32),candidate_ids\n\ndef metrics(sample,truth,owners,candidate_ids,prob,threshold,split):\n    selected=np.flatnonzero(sample.split.to_numpy()==split)\n    pred={int(i):set() for i in selected}\n    for i,c,p in zip(owners,candidate_ids,prob):\n        if int(i) in pred and p>=threshold: pred[int(i)].add(c)\n    scores=[]; tp=fp=fn=singletons=singleton_correct=0\n    for i in selected:\n        gt=truth[sample.iloc[i].entity_id]; pr=pred[int(i)]\n        a=len(gt&pr); b=len(pr-gt); c=len(gt-pr)\n        tp+=a; fp+=b; fn+=c\n        if not gt:\n            singletons+=1; singleton_correct+=int(not pr); scores.append(float(not pr))\n        else: scores.append(1.25*a/(1.25*a+b+.25*c))\n    return dict(macro_F05=float(np.mean(scores)),correct_accepted=tp,incorrect_accepted=fp,\n                missed_true_links=fn,singletons_correct=singleton_correct,singletons=singletons)\n\ndef fit_and_evaluate(work,sample,truth,ids,scores,records):\n    from importlib.metadata import version\n    x,y,owners,candidate_ids=make_pairs(sample,truth,ids,scores,records,k=20)\n    splits=sample.split.to_numpy()[owners]; train=splits==\'train\'\n    assert len(np.unique(y[train]))==2,\'Need positive and negative training pairs.\'\n    log(f\'Training new matcher: {train.sum():,} pairs, {y[train].sum():,} true matches\')\n    model=HistGradientBoostingClassifier(learning_rate=.08,max_iter=250,max_leaf_nodes=31,\n              l2_regularization=1.,early_stopping=False,random_state=2026)\n    with threadpool_limits(limits=4):\n        model.fit(x[train],y[train]); prob=model.predict_proba(x)[:,1]\n    # Only tuning businesses choose the threshold. Evaluation is read once afterwards.\n    tuning=[]\n    for t in np.linspace(.05,.95,91):\n        tuning.append({\'threshold\':float(t),**metrics(sample,truth,owners,candidate_ids,prob,t,\'tune\')})\n    best=max(tuning,key=lambda d:(d[\'macro_F05\'],d[\'threshold\']))\n    cutoff=best[\'threshold\']\n    evaluation=metrics(sample,truth,owners,candidate_ids,prob,cutoff,\'evaluation\')\n    selected=np.flatnonzero(sample.split.to_numpy()==\'evaluation\')\n    search_missed=0\n    for i in selected:\n        retrieved={records[int(r)][0] for r in ids[i,:20] if r>=0}\n        search_missed+=len(truth[sample.iloc[i].entity_id]-retrieved)\n    evaluation[\'missed_by_search\']=search_missed\n    evaluation[\'retrieved_but_rejected\']=evaluation[\'missed_true_links\']-search_missed\n    result={\'version\':VERSION,\'model_id\':MODEL_ID,\'revision\':REVISION,\'candidate_k\':20,\n            \'ann_nprobe\':64,\'ann_overfetch\':200,\'threshold\':cutoff,\'tuning\':best,\n            \'evaluation\':evaluation,\'features\':MATCH_FEATURES,\n            \'scope\':\'Full training S2/S3; 3000 S1 train, 500 tune, 500 evaluation; no test predictions.\',\n            \'versions\':{p:version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'faiss-cpu\',\'torch\',\'transformers\']}}\n    atomic_json(work/\'experiment_results.json\',result)\n    pd.DataFrame(tuning).to_csv(work/\'threshold_tuning.csv\',index=False)\n    with (work/\'matcher.pkl\').open(\'wb\') as f: pickle.dump({\'model\':model,\'threshold\':cutoff,\'features\':MATCH_FEATURES},f)\n    # Save diagnostics, clearly labeled TRAINING evaluation, not a portal submission.\n    diag=pd.DataFrame({\'source1_id\':sample.entity_id.to_numpy()[owners],\'candidate_id\':candidate_ids,\n                       \'split\':splits,\'label\':y,\'score\':prob})\n    diag[diag.split!=\'train\'].to_csv(work/\'heldout_pair_scores.tsv\',sep=\'\\t\',index=False)\n    log(json.dumps(result,indent=2))\n    return result\n'
RUNNER_SOURCE='"""Run the evaluated E5 + matcher pipeline on the complete challenge test set.\n\nNo fitting or threshold selection occurs here. Candidate exports contain exactly\nall final candidate pairs passed to the saved matcher.\n"""\nimport argparse, csv, gc, hashlib, json, os, pickle, shutil, subprocess, sys, time, zipfile\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom threadpoolctl import threadpool_limits\nimport embedding_core as core\n\nVERSION=\'embedding-submission-v1\'\nPREDICT_CHUNK=2000\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(4*1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef locate_model(explicit=None):\n    if explicit:\n        candidates=[Path(explicit)]\n    else:\n        candidates=[]\n        local=Path(\'/kaggle/working/full_pool_embedding_v1\')\n        if (local/\'matcher.pkl\').exists():candidates.append(local)\n        for path in Path(\'/kaggle/input\').rglob(\'experiment_results.json\'):\n            parent=path.parent\n            if (parent/\'matcher.pkl\').exists():candidates.append(parent)\n    valid=[]\n    for path in sorted(set(candidates)):\n        result=path/\'experiment_results.json\'\n        if result.exists() and (path/\'matcher.pkl\').exists():\n            try:meta=json.loads(result.read_text())\n            except (ValueError,OSError):continue\n            if meta.get(\'version\')==\'full-pool-e5-v1\' and meta.get(\'model_id\')==core.MODEL_ID:valid.append(path)\n    if len(valid)!=1:\n        raise RuntimeError(\'Attach the completed full-pool experiment notebook OUTPUT as an input, \'\n          \'then set MODEL_DIR to the folder containing matcher.pkl and experiment_results.json. \'\n          f\'Found {len(valid)} matching folders: {[str(p) for p in valid]}\')\n    return valid[0]\n\ndef load_model(model_dir):\n    import sklearn\n    meta=json.loads((model_dir/\'experiment_results.json\').read_text())\n    assert meta[\'revision\']==core.REVISION and meta[\'features\']==core.MATCH_FEATURES\n    assert meta[\'candidate_k\']==20 and meta[\'ann_nprobe\']==64 and meta[\'ann_overfetch\']==200\n    assert sklearn.__version__==meta[\'versions\'][\'scikit-learn\'], (\n        \'Install scikit-learn==\'+meta[\'versions\'][\'scikit-learn\']+\' before loading this model.\')\n    # This must be YOUR trusted matcher produced by the previous notebook.\n    with (model_dir/\'matcher.pkl\').open(\'rb\') as f:bundle=pickle.load(f)\n    assert bundle[\'features\']==core.MATCH_FEATURES\n    assert abs(float(bundle[\'threshold\'])-float(meta[\'threshold\']))<1e-10\n    assert bundle[\'model\'].n_features_in_==len(core.MATCH_FEATURES)\n    assert list(bundle[\'model\'].classes_)==[0,1]\n    return bundle,meta\n\ndef initialize(data,work,model_dir):\n    data=Path(data);work=Path(work);model_dir=Path(model_dir)\n    work.mkdir(parents=True,exist_ok=True)\n    files=[data/f\'test/test_source{i}.tsv\' for i in (1,2,3)]\n    validator=data.parent/\'utils/validate_submission.py\'\n    for path in files+[validator]:\n        if not path.exists():raise FileNotFoundError(path)\n    bundle,meta=load_model(model_dir)\n    core.log(\'Checking input fingerprints and saved model...\')\n    config={\'version\':VERSION,\'inputs\':{p.name:sha256(p) for p in files},\n      \'matcher_sha256\':sha256(model_dir/\'matcher.pkl\'),\'model_id\':core.MODEL_ID,\n      \'revision\':core.REVISION,\'max_length\':256,\'dim\':core.DIM,\n      \'candidate_k\':20,\'nprobe\':64,\'overfetch\':200,\'threshold\':float(bundle[\'threshold\']),\n      \'prediction_chunk\':PREDICT_CHUNK,\n      \'core_sha256\':sha256(Path(core.__file__)),\'runner_sha256\':sha256(Path(__file__))}\n    cp=work/\'submission_config.json\'\n    if cp.exists():assert json.loads(cp.read_text())==config,\'Inputs, model or code changed. Use a NEW WORK directory.\'\n    else:core.atomic_json(cp,config)\n    artifact=work/\'model\';artifact.mkdir(exist_ok=True)\n    for name in [\'matcher.pkl\',\'experiment_results.json\']:\n        dest=artifact/name\n        if dest.resolve()!=(model_dir/name).resolve():shutil.copy2(model_dir/name,dest)\n    core.log(f"Saved matcher cutoff: {bundle[\'threshold\']:.3f}; 20 candidates; 64 cells / 200 intermediate neighbors")\n    return bundle,meta,config\n\nclass TestSearcher:\n    def __init__(self,work):\n        import faiss\n        self.work=work;self.faiss=faiss\n        self.total=json.loads((work/\'encoding_complete.json\').read_text())[\'total\']\n        self.vectors=np.memmap(work/\'vectors.f16\',mode=\'r\',dtype=np.float16,shape=(self.total,core.DIM))\n        self.db=sqlite3.connect(f\'file:{(work/"records.sqlite").resolve()}?mode=ro\',uri=True)\n        self.db.execute(\'PRAGMA cache_size=-65536\')\n        self.indexes={}\n        for item in json.loads((work/\'index_catalog.json\').read_text()):\n            index=faiss.read_index(str(work/item[\'file\']))\n            assert index.ntotal==item[\'count\']\n            if hasattr(index,\'nprobe\'):index.nprobe=min(64,index.nlist)\n            self.indexes[item[\'country\']]=index\n    def search(self,frame,q):\n        """Same approximate search and float16-vector cosine reranking as evaluation."""\n        ids=np.full((len(frame),20),-1,np.int64)\n        scores=np.full((len(frame),20),-1,np.float32)\n        countries=frame.country.map(core.norm).to_numpy()\n        for country in np.unique(countries):\n            qi=np.flatnonzero(countries==country)\n            index=self.indexes.get(country)\n            if index is None:continue\n            for start in range(0,len(qi),64):\n                indices=qi[start:start+64]\n                _,approx=index.search(np.ascontiguousarray(q[indices],dtype=np.float32),200)\n                # Batch gathers reduce Python/disk overhead; re-ranking remains cosine.\n                valid=approx>=0\n                safe=np.maximum(approx,0)\n                x=np.asarray(self.vectors[safe],dtype=np.float32)\n                x/=np.maximum(np.linalg.norm(x,axis=2,keepdims=True),1e-12)\n                cos=np.einsum(\'bkd,bd->bk\',x,q[indices],optimize=False)\n                cos[~valid]=-np.inf\n                for local,i in enumerate(indices):\n                    positions=np.flatnonzero(valid[local])\n                    if not len(positions):continue\n                    order=np.lexsort((approx[local,positions],-cos[local,positions]))\n                    chosen=positions[order[:20]]\n                    ids[i,:len(chosen)]=approx[local,chosen]\n                    scores[i,:len(chosen)]=cos[local,chosen]\n                del x\n        return ids,scores\n    def fetch(self,ids):\n        valid=np.unique(ids[ids>=0]);rows={}\n        for start in range(0,len(valid),800):\n            batch=[int(x) for x in valid[start:start+800]]\n            sql=\'SELECT * FROM records WHERE rid IN (\'+\',\'.join(\'?\'*len(batch))+\')\'\n            for row in self.db.execute(sql,batch):rows[row[0]]=tuple(row[1:])\n        assert len(rows)==len(valid),\'Candidate IDs missing from test record store.\'\n        return rows\n    def close(self):\n        self.db.close();self.indexes.clear();del self.vectors\n\n# sqlite is separate from core to keep this module directly runnable.\nimport sqlite3\n\ndef score_candidates(frame,ids,scores,records,bundle):\n    candidates=[[] for _ in range(len(frame))];accepted=[[] for _ in range(len(frame))]\n    features=[];owners=[];pair_ids=[]\n    for i,left in enumerate(frame[core.COLS].itertuples(index=False,name=None)):\n        seen=set()\n        for rank,rid in enumerate(ids[i]):\n            if rid<0:continue\n            right=records[int(rid)]\n            assert core.norm(left[3])==core.norm(right[3]),\'Country mismatch in retrieval.\'\n            assert right[0].startswith((\'S2-\',\'S3-\'))\n            assert right[0] not in seen,\'Duplicate candidate entity ID in test input/retrieval.\'\n            seen.add(right[0]);candidates[i].append(right[0])\n            features.append(core.features(left,right)+[float(scores[i,rank])])\n            owners.append(i);pair_ids.append(right[0])\n    if features:\n        x=np.asarray(features,np.float32)\n        assert np.isfinite(x).all()\n        prob=bundle[\'model\'].predict_proba(x)[:,1]\n        assert np.isfinite(prob).all()\n        for owner,cid,score in zip(owners,pair_ids,prob):\n            if score>=bundle[\'threshold\']:accepted[owner].append(cid)\n    assert sum(map(len,candidates))==len(features)\n    for cand,match in zip(candidates,accepted):\n        assert len(cand)<=20 and set(match)<=set(cand)\n    return candidates,accepted\n\ndef write_shard(path,frame,candidates,accepted,start):\n    staging=path.with_name(path.name+\'.tmp\');staging.mkdir(parents=True,exist_ok=True)\n    for name,values,col in [(\'candidate_pairs.tsv\',candidates,\'candidate_entity_ids\'),\n                            (\'matching_results.tsv\',accepted,\'matched_entity_ids\')]:\n        with (staging/name).open(\'w\',encoding=\'utf-8\',newline=\'\') as f:\n            writer=csv.writer(f,delimiter=\'\\t\',lineterminator=\'\\n\')\n            writer.writerow([\'source1_entity_id\',col])\n            for sid,links in zip(frame.entity_id,values):writer.writerow([sid,\',\'.join(links)])\n            f.flush();os.fsync(f.fileno())\n    stats={\'start\':start,\'rows\':len(frame),\'first\':str(frame.entity_id.iloc[0]),\'last\':str(frame.entity_id.iloc[-1]),\n       \'candidate_pairs\':sum(map(len,candidates)),\'predicted_links\':sum(map(len,accepted)),\n       \'empty_candidates\':sum(not x for x in candidates),\'empty_matches\':sum(not x for x in accepted),\n       \'candidate_histogram\':np.bincount([len(x) for x in candidates],minlength=21).tolist()}\n    core.atomic_json(staging/\'stats.json\',stats)\n    os.replace(staging,path)\n    return stats\n\ndef predict(data,work,bundle,encoder):\n    shards=work/\'shards\';shards.mkdir(exist_ok=True)\n    searcher=TestSearcher(work)\n    counts={};start=0;started=time.time();new=0\n    total_s1=sum(len(c) for c in core.read_chunks(data/\'test/test_source1.tsv\',[\'entity_id\']))\n    core.log(f\'Predicting all {total_s1:,} test Source 1 businesses; output checkpoint every {PREDICT_CHUNK:,}.\')\n    try:\n        with threadpool_limits(limits=4):\n            searcher.faiss.omp_set_num_threads(min(4,os.cpu_count() or 2))\n            for frame in pd.read_csv(data/\'test/test_source1.tsv\',sep=\'\\t\',dtype=str,\n                                      keep_default_na=False,usecols=core.COLS,chunksize=PREDICT_CHUNK):\n                frame=frame[core.COLS].reset_index(drop=True)\n                path=shards/f\'{start:010d}\'\n                if path.exists():\n                    stats=json.loads((path/\'stats.json\').read_text())\n                    assert stats[\'start\']==start and stats[\'rows\']==len(frame)\n                    assert stats[\'first\']==frame.entity_id.iloc[0] and stats[\'last\']==frame.entity_id.iloc[-1]\n                    assert all((path/n).exists() for n in [\'candidate_pairs.tsv\',\'matching_results.tsv\'])\n                else:\n                    if shutil.disk_usage(work).free<1.5*2**30:raise RuntimeError(\'Low disk. Completed shards saved; free space before rerunning.\')\n                    q=encoder.encode(frame)\n                    ids,scores=searcher.search(frame,q)\n                    records=searcher.fetch(ids)\n                    cand,accepted=score_candidates(frame,ids,scores,records,bundle)\n                    stats=write_shard(path,frame,cand,accepted,start)\n                    new+=len(frame)\n                    del records,q,ids,scores,cand,accepted\n                    if new==len(frame) or (start+len(frame))%10000==0:\n                        rate=new/max(time.time()-started,1)\n                        eta=(total_s1-start-len(frame))/max(rate,1)/60\n                        core.log(f\'{start+len(frame):,}/{total_s1:,} businesses; {rate:.1f}/sec; remaining prediction ~{eta:.1f} min\')\n                for k in [\'rows\',\'candidate_pairs\',\'predicted_links\',\'empty_candidates\',\'empty_matches\']:\n                    counts[k]=counts.get(k,0)+stats[k]\n                start+=len(frame)\n    finally:searcher.close();gc.collect()\n    assert start==total_s1\n    core.atomic_json(work/\'prediction_complete.json\',counts)\n    core.log(\'All test businesses predicted: \'+json.dumps(counts))\n    return counts\n\ndef merge_outputs(data,work):\n    counts=json.loads((work/\'prediction_complete.json\').read_text())\n    out=work/\'output\';out.mkdir(exist_ok=True)\n    paths=sorted(p for p in (work/\'shards\').iterdir() if p.is_dir() and p.name.isdigit())\n    expected=0;hist=np.zeros(21,np.int64)\n    for p in paths:\n        s=json.loads((p/\'stats.json\').read_text())\n        assert s[\'start\']==expected\n        expected+=s[\'rows\'];hist+=s[\'candidate_histogram\']\n    assert expected==counts[\'rows\']\n    for filename,second in [(\'matching_results.tsv\',\'matched_entity_ids\'),(\'candidate_pairs.tsv\',\'candidate_entity_ids\')]:\n        path=out/filename;tmp=Path(str(path)+\'.tmp\');rows=0\n        with tmp.open(\'wb\') as f:\n            header=f\'source1_entity_id\\t{second}\\n\'.encode()\n            f.write(header)\n            for p in paths:\n                with (p/filename).open(\'rb\') as src:\n                    assert src.readline()==header\n                    for line in src:\n                        f.write(line);rows+=1\n        assert rows==expected\n        os.replace(tmp,path)\n    counts[\'mean_candidates\']=counts[\'candidate_pairs\']/counts[\'rows\']\n    counts[\'candidate_histogram\']=hist.tolist()\n    core.atomic_json(out/\'submission_summary.json\',counts)\n    core.log(\'Merged submission files: \'+json.dumps(counts))\n    return out\n\ndef validate(data,work):\n    validator=data.parent/\'utils/validate_submission.py\'\n    out=work/\'output\'\n    command=[sys.executable,str(validator),\'--matching\',str(out/\'matching_results.tsv\'),\n             \'--candidate\',str(out/\'candidate_pairs.tsv\'),\'--test-dir\',str(data/\'test\'),\'--check-ids\']\n    core.log(\'Running official validator, including test ID existence checks...\')\n    with (out/\'validation.log\').open(\'w\') as log:\n        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)\n        for line in proc.stdout:\n            print(line,end=\'\',flush=True);log.write(line);log.flush()\n        code=proc.wait()\n    if code:raise RuntimeError(f\'Official validator failed (exit {code}). Do not submit. See validation.log.\')\n    core.atomic_json(out/\'validated.json\',{\'matching_sha256\':sha256(out/\'matching_results.tsv\'),\n               \'candidate_sha256\':sha256(out/\'candidate_pairs.tsv\'),\'check_ids\':True})\n    core.log(\'VALIDATOR PASSED. Portal upload: \'+str(out/\'matching_results.tsv\'))\n\ndef package(work,meta):\n    from importlib.metadata import version\n    out=work/\'output\';assert (out/\'validated.json\').exists()\n    validation=json.loads((out/\'validated.json\').read_text())\n    assert validation[\'matching_sha256\']==sha256(out/\'matching_results.tsv\')\n    assert validation[\'candidate_sha256\']==sha256(out/\'candidate_pairs.tsv\')\n    (work/\'requirements.txt\').write_text(\'\\n\'.join(f\'{p}=={version(p)}\' for p in\n             [\'torch\',\'transformers\',\'numpy\',\'pandas\',\'scikit-learn\',\'faiss-cpu\',\'safetensors\',\'sentencepiece\'])+\'\\n\')\n    (work/\'README.md\').write_text(\'\'\'# Embedding submission v1\nUpload `output/matching_results.tsv` to the leaderboard portal.\nRetain `output/candidate_pairs.tsv`, the model, code, and methodology for final review.\nThe candidate file contains exactly every final pair evaluated by the classifier.\n\n## Reproduce predictions\nInstall requirements in a compatible GPU environment. Public E5 weights are pinned by revision.\nRun: `python submit_pipeline.py --data /path/to/student_resource/dataset --model-dir ./model --work ./rerun --stage all`\nThe input dataset and official validator must be provided separately. No business lookups occur.\nAll test S1 entities receive one row, including empty predictions. Country matching is enforced.\n\n## Reproduce training\nRun `training_experiment.ipynb` on the supplied training dataset to recreate the sampled\ntraining experiment, matcher and cutoff. Its dataset and working paths are configurable.\nThe trained matcher in `model/matcher.pkl` comes from that experiment, not test labels.\nOnly load your own trusted pickle models.\n\n## Checkpoints\nThis archive excludes large vector and index caches. The complete saved Kaggle output\ncontains those, plus completed prediction shards. Rerunning against an unchanged working\nfolder resumes completed work. A discarded unsaved session cannot be resumed.\n\n## Final package\nThis archive preserves submission evidence and runnable code. Before final challenge\nclosure, transfer methodology into the organizer\'s Documentation_template.md and check\nany organizer-specific archive naming or additional requirements.\n\'\'\')\n    (work/\'methodology.md\').write_text(f\'\'\'# Methodology — embedding submission v1\nNames and addresses are NFKC/whitespace normalized, preserving Unicode. E5 encodes\n`query: business name: NAME; address: ADDRESS` with attention-masked mean pooling and\nL2 normalization, maximum 256 tokens. Model: {core.MODEL_ID}; revision: {core.REVISION}.\nPretrained weights are fixed, not fine-tuned. Business data comes only from the challenge.\n\nTest S2/S3 vectors form independent country-specific IVF-PQ indexes (up to 2048 cells,\n48 subquantizers, 8 bits each). Quantizers use up to 110000 unlabeled target vectors\nper country, seed 2026. Search probes 64 cells, retrieves up to 200 approximate candidates,\nthen reranks by cosine against stored float16 vectors and keeps at most 20.\nOnly these final 20 are passed to the matcher and written to candidate_pairs.tsv.\nIntermediate retrieval counts and code are disclosed here for auditability.\n\nThe HistGradientBoostingClassifier uses the 18 lexical features defined in embedding_core.py\nand embedding cosine similarity. It was trained on 3000 sampled S1 businesses (60000 pairs),\nwith 500 tuning and 500 evaluation businesses. Threshold: {meta[\'threshold\']}.\nTraining targets include the entire provided train S2/S3 pool. No truth matches are injected.\nLocal evaluation macro F0.5: {meta[\'evaluation\'][\'macro_F05\']:.6f}; this is not a test leaderboard score.\nTest predictions use the saved model and cutoff without retraining or threshold changes.\n\nSource 1 is processed in batches and files are assembled in input order. The official\nvalidator runs with --check-ids. Empty match lists are retained. France is present in test\nbut absent from labeled training; country-specific French accuracy has not been validated.\nThe country index code derives countries dynamically, including France.\n\nSee output/submission_summary.json for actual candidate counts, and requirements.txt for versions.\n\'\'\')\n    # A small ZIP is easy to download even if the browser previews TSV files.\n    with zipfile.ZipFile(out/\'matching_results_download.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n        z.write(out/\'matching_results.tsv\',\'matching_results.tsv\')\n    archive=work.parent/\'embedding_submission_v1_backup.zip\'\n    members=[\'README.md\',\'methodology.md\',\'requirements.txt\',\'submission_config.json\',\n      \'embedding_core.py\',\'submit_pipeline.py\',\'training_experiment.ipynb\',\n      \'model/matcher.pkl\',\'model/experiment_results.json\',\'output/matching_results.tsv\',\n      \'output/candidate_pairs.tsv\',\'output/submission_summary.json\',\'output/validation.log\',\'output/validated.json\']\n    for name in members:\n        assert (work/name).exists(),f\'Missing backup member: {name}\'\n    with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for name in members:z.write(work/name,name)\n    core.log(\'Downloadable matches ZIP: \'+str(out/\'matching_results_download.zip\'))\n    core.log(\'Retain backup: \'+str(archive))\n    return archive\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--data\',type=Path,required=True)\n    parser.add_argument(\'--work\',type=Path,required=True)\n    parser.add_argument(\'--model-dir\',type=Path,required=True)\n    parser.add_argument(\'--stage\',choices=[\'all\',\'encode\',\'index\',\'predict\',\'finalize\'],default=\'all\')\n    parser.add_argument(\'--batch-size\',type=int,default=64)\n    args=parser.parse_args();data,work=args.data,args.work\n    bundle,meta,config=initialize(data,work,args.model_dir)\n    if args.stage in (\'all\',\'encode\'):\n        encoder=core.Encoder(args.batch_size)\n        core.encode_pool(data,work,encoder,split=\'test\')\n        del encoder;gc.collect()\n        import torch;torch.cuda.empty_cache()\n    if args.stage in (\'all\',\'index\'):core.build_indexes(work)\n    if args.stage in (\'all\',\'predict\'):\n        encoder=core.Encoder(args.batch_size)\n        predict(data,work,bundle,encoder)\n        del encoder;gc.collect()\n        import torch;torch.cuda.empty_cache()\n    if args.stage in (\'all\',\'finalize\'):\n        merge_outputs(data,work);validate(data,work);package(work,meta)\n\nif __name__==\'__main__\':main()\n'
TRAINING_NOTEBOOK='{\n "cells": [\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "# Amazon ML \\u2014 full-pool embedding experiment\\n",\n    "\\n",\n    "**Purpose:** check the promising embedding pilot against every training Source 2/3 record, then train a new matching classifier. This is an experiment, not a portal submission.\\n",\n    "\\n",\n    "1. Encode all training S2/S3 records with the same combined-name-and-address E5 model as the pilot.\\n",\n    "2. Build a compressed FAISS IVF-PQ index for each country. Search selected cells, rather than comparing every pair.\\n",\n    "3. Retrieve up to 200 approximate neighbors, reorder these by cosine similarity, and keep **20 candidates per Source 1 business for the matcher**. Measure retrieval at 10/20/50 too.\\n",\n    "4. Train on 3,000 Source 1 businesses; choose the acceptance threshold on 500; report results on another 500. All target records participate in retrieval, without using ground truth to select them.\\n",\n    "\\n",\n    "**Kaggle setup:** new notebook; attach the challenge dataset; enable a GPU and Internet; use a fresh output folder. The old 0.756 submission remains separate. Allow several hours: actual time depends on GPU, CPU and text lengths. Progress logs estimate encoding time after work starts.\\n",\n    "\\n",\n    "The model runs locally. No business records go to an external API. This uses the pilot\'s fixed MIT-licensed E5 revision. It requires roughly 13 GiB free disk initially, including 7.4 GiB for vectors; the notebook checks space. Do not run the old full pipeline again in the same session.\\n",\n    "\\n",\n    "**Saving:** choose **Save Version \\u2192 Save & Run All**. Checkpoints survive a cell rerun in the same filesystem; they do not survive an unsaved, discarded Kaggle session. Keep the saved version\'s outputs. Restoring a run requires the entire work folder, not just the small results ZIP.\\n"\n   ],\n   "id": "cell-00"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "%pip install -q \\"faiss-cpu==1.11.0\\" \\"transformers>=4.48,<6\\" safetensors sentencepiece\\n"\n   ],\n   "execution_count": null,\n   "outputs": [],\n   "id": "cell-01"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "from pathlib import Path\\n",\n    "import os, sys, shutil, torch\\n",\n    "DATA = Path(\'/kaggle/input/datasets/spoortimorabad/ml-challenge-dataset/student_resource/dataset\')\\n",\n    "WORK = Path(\'/kaggle/working/full_pool_embedding_v1\')\\n",\n    "BATCH_SIZE = 64\\n",\n    "assert torch.cuda.is_available(), \'Enable a GPU in Kaggle Settings before running.\'\\n",\n    "assert (DATA/\'train/train_source1.tsv\').exists(), \'Update DATA to your attached dataset directory.\'\\n",\n    "WORK.mkdir(parents=True, exist_ok=True)\\n",\n    "os.environ[\'TOKENIZERS_PARALLELISM\']=\'false\'\\n",\n    "print(\'GPU:\', torch.cuda.get_device_name(0))\\n",\n    "print(\'Free disk GiB:\', round(shutil.disk_usage(WORK).free/2**30, 2))\\n"\n   ],\n   "execution_count": null,\n   "outputs": [],\n   "id": "cell-02"\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## Load the self-contained pipeline\\n",\n    "No variables or files from the old notebook are required. If you change model settings or code, use a new WORK folder to avoid mixing caches."\n   ],\n   "id": "cell-03"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "PIPELINE_SOURCE = \'import os, json, time, gc, sqlite3, hashlib, shutil, pickle, re, unicodedata\\\\nfrom pathlib import Path\\\\nfrom functools import lru_cache\\\\nimport numpy as np\\\\nimport pandas as pd\\\\nfrom sklearn.ensemble import HistGradientBoostingClassifier\\\\nfrom threadpoolctl import threadpool_limits\\\\nCOLS = [\\\\\'entity_id\\\\\', \\\\\'business_name\\\\\', \\\\\'business_address\\\\\', \\\\\'country\\\\\']\\\\n\\\\nLEGAL = set(\\\\\'inc incorporated llc llp ltd limited pvt private corp corporation company co plc sarl sas eurl\\\\\'.split())\\\\n\\\\nADDR_ALIAS = {\\\\\'road\\\\\': \\\\\'rd\\\\\', \\\\\'street\\\\\': \\\\\'st\\\\\', \\\\\'avenue\\\\\': \\\\\'ave\\\\\', \\\\\'drive\\\\\': \\\\\'dr\\\\\', \\\\\'lane\\\\\': \\\\\'ln\\\\\', \\\\\'boulevard\\\\\': \\\\\'blvd\\\\\', \\\\\'apartment\\\\\': \\\\\'apt\\\\\', \\\\\'suite\\\\\': \\\\\'ste\\\\\'}\\\\n\\\\nFEATURES = [\\\\\'name_exact\\\\\', \\\\\'core_name_exact\\\\\', \\\\\'name_trigram_jaccard\\\\\', \\\\\'name_token_jaccard\\\\\', \\\\\'name_token_containment\\\\\', \\\\\'name_length_ratio\\\\\', \\\\\'address_exact\\\\\', \\\\\'address_trigram_jaccard\\\\\', \\\\\'address_token_jaccard\\\\\', \\\\\'address_token_containment\\\\\', \\\\\'address_missing\\\\\', \\\\\'number_jaccard\\\\\', \\\\\'number_disjoint\\\\\', \\\\\'unit_equal\\\\\', \\\\\'unit_conflict\\\\\', \\\\\'country_equal\\\\\', \\\\\'name_token_count\\\\\', \\\\\'candidate_name_token_count\\\\\']\\\\n\\\\ndef log(s):\\\\n    print(s, flush=True)\\\\n\\\\ndef norm(s):\\\\n    s = unicodedata.normalize(\\\\\'NFKC\\\\\', str(s)).casefold()\\\\n    return \\\\\' \\\\\'.join(re.findall(\\\\\'[^\\\\\\\\\\\\\\\\W_]+\\\\\', s))\\\\n\\\\ndef canonical_address(s):\\\\n    return \\\\\' \\\\\'.join((ADDR_ALIAS.get(t, t) for t in norm(s).split() if t not in {\\\\\'null\\\\\', \\\\\'none\\\\\'}))\\\\n\\\\ndef grams(s):\\\\n    return frozenset((s[i:i + 3] for i in range(max(0, len(s) - 2))))\\\\n\\\\ndef unit_id(s):\\\\n    m = re.search(\\\\\'\\\\\\\\\\\\\\\\b(?:unit|suite|ste|apt|apartment|flat)\\\\\\\\\\\\\\\\s*(?:no\\\\\\\\\\\\\\\\.?\\\\\\\\\\\\\\\\s*)?[:#-]?\\\\\\\\\\\\\\\\s*([\\\\\\\\\\\\\\\\w]+)\\\\\', unicodedata.normalize(\\\\\'NFKC\\\\\', s).casefold())\\\\n    return m.group(1) if m else \\\\\'\\\\\'\\\\n\\\\n@lru_cache(maxsize=30000)\\\\ndef prepared(name, address, country):\\\\n    n = norm(name)\\\\n    a = canonical_address(address)\\\\n    nt, at = (frozenset(n.split()), frozenset(a.split()))\\\\n    core = tuple(sorted((t for t in nt if t not in LEGAL)))\\\\n    nums = frozenset(re.findall(\\\\\'\\\\\\\\\\\\\\\\d+[a-z]*\\\\\', a))\\\\n    return (n, a, nt, at, core, nums, grams(n), grams(a), unit_id(address), norm(country))\\\\n\\\\ndef jaccard(a, b):\\\\n    return len(a & b) / len(a | b) if a or b else 0.0\\\\n\\\\ndef containment(a, b):\\\\n    return len(a & b) / min(len(a), len(b)) if a and b else 0.0\\\\n\\\\ndef features(left, right):\\\\n    n, a, nt, at, c, nums, ng, ag, u, country = prepared(*left[1:])\\\\n    m, b, mt, bt, d, other_nums, mg, bg, v, other_country = prepared(*right[1:])\\\\n    return [float(bool(n) and n == m), float(bool(c) and c == d), jaccard(ng, mg), jaccard(nt, mt), containment(nt, mt), min(len(n), len(m)) / max(len(n), len(m), 1), float(bool(a) and a == b), jaccard(ag, bg), jaccard(at, bt), containment(at, bt), float(not a or not b), jaccard(nums, other_nums), float(bool(nums) and bool(other_nums) and (not nums & other_nums)), float(bool(u) and bool(v) and (u == v)), float(bool(u) and bool(v) and (u != v)), float(bool(country) and country == other_country), len(nt), len(mt)]\\\\n\\\\ndef read_chunks(path, cols=COLS):\\\\n    return pd.read_csv(path, sep=\\\\\'\\\\\\\\t\\\\\', dtype=str, keep_default_na=False, usecols=cols, chunksize=100000)\\\\n\\\\ndef reproduce_reference_sample(data):\\\\n    from sklearn.model_selection import train_test_split\\\\n    rng = np.random.default_rng(2026)\\\\n    sample = pd.DataFrame()\\\\n    for chunk in read_chunks(data / \\\\\'train/train_source1.tsv\\\\\'):\\\\n        chunk = chunk[COLS].copy()\\\\n        chunk[\\\\\'_priority\\\\\'] = rng.random(len(chunk))\\\\n        sample = pd.concat([sample, chunk], ignore_index=True).nsmallest(4000, \\\\\'_priority\\\\\')\\\\n    sample = sample.drop(columns=\\\\\'_priority\\\\\').reset_index(drop=True)\\\\n    assert len(sample) == 4000 and sample.entity_id.is_unique\\\\n    ids = set(sample.entity_id)\\\\n    truth = {}\\\\n    for chunk in read_chunks(data / \\\\\'train/train_ground_truth.tsv\\\\\', [\\\\\'source1_entity_id\\\\\', \\\\\'matched_entity_ids\\\\\']):\\\\n        for row in chunk[chunk.source1_entity_id.isin(ids)].itertuples(index=False):\\\\n            truth[row.source1_entity_id] = set((x.strip() for x in row.matched_entity_ids.split(\\\\\',\\\\\') if x.strip()))\\\\n    assert set(truth) == ids\\\\n    strata = sample.country + \\\\\'|\\\\\' + sample.entity_id.map(lambda x: str(not truth[x]))\\\\n    train_idx, held = train_test_split(np.arange(4000), test_size=0.25, random_state=2026, stratify=strata)\\\\n    tune_idx, eval_idx = train_test_split(held, test_size=0.5, random_state=2027, stratify=strata.iloc[held])\\\\n    sample[\\\\\'split\\\\\'] = \\\\\'train\\\\\'\\\\n    sample.loc[tune_idx, \\\\\'split\\\\\'] = \\\\\'tune\\\\\'\\\\n    sample.loc[eval_idx, \\\\\'split\\\\\'] = \\\\\'evaluation\\\\\'\\\\n    return (sample, truth)\\\\n\\\\ndef clean_embedding_text(s):\\\\n    return \\\\\' \\\\\'.join(unicodedata.normalize(\\\\\'NFKC\\\\\', str(s)).split())\\\\n\\\\ndef text_view(frame, view):\\\\n    if view == \\\\\'name\\\\\':\\\\n        return [\\\\\'query: \\\\\' + clean_embedding_text(x) for x in frame.business_name]\\\\n    return [\\\\\'query: business name: \\\\\' + clean_embedding_text(n) + \\\\\'; address: \\\\\' + clean_embedding_text(a) for n, a in zip(frame.business_name, frame.business_address)]\\\\n\\\\nMODEL_ID=\\\\\'intfloat/multilingual-e5-small\\\\\'\\\\nREVISION=\\\\\'614241f622f53c4eeff9890bdc4f31cfecc418b3\\\\\'\\\\nDIM=384\\\\nVERSION=\\\\\'full-pool-e5-v1\\\\\'\\\\n\\\\ndef atomic_json(path, value):\\\\n    tmp=Path(str(path)+\\\\\'.tmp\\\\\')\\\\n    tmp.write_text(json.dumps(value,indent=2))\\\\n    os.replace(tmp,path)\\\\n\\\\ndef initialize(data,work):\\\\n    work=Path(work); data=Path(data); work.mkdir(parents=True,exist_ok=True)\\\\n    files=[data/f\\\\\'train/train_source{i}.tsv\\\\\' for i in (1,2,3)]+[data/\\\\\'train/train_ground_truth.tsv\\\\\']\\\\n    for p in files:\\\\n        assert p.exists(),f\\\\\'Missing dataset: {p}\\\\\'\\\\n    config={\\\\\'version\\\\\':VERSION,\\\\\'model\\\\\':MODEL_ID,\\\\\'revision\\\\\':REVISION,\\\\\'dim\\\\\':DIM,\\\\n            \\\\\'max_length\\\\\':256,\\\\\'text\\\\\':\\\\\'query: business name: NAME; address: ADDRESS\\\\\',\\\\n            \\\\\'files\\\\\':[(str(p),p.stat().st_size) for p in files],\\\\n            \\\\\'nlist_max\\\\\':2048,\\\\\'pq_m\\\\\':48,\\\\\'pq_bits\\\\\':8,\\\\\'nprobe\\\\\':64,\\\\\'overfetch\\\\\':200,\\\\n            \\\\\'candidate_k\\\\\':20,\\\\\'sample_seed\\\\\':2026,\\\\\'sample_size\\\\\':4000}\\\\n    cp=work/\\\\\'configuration.json\\\\\'\\\\n    if cp.exists():\\\\n        assert json.loads(cp.read_text())==json.loads(json.dumps(config)), \\\\\'Configuration changed. Use a new WORK folder.\\\\\'\\\\n    else: atomic_json(cp,config)\\\\n    log(f\\\\\'Free disk: {shutil.disk_usage(work).free/2**30:.1f} GiB\\\\\')\\\\n    return data,work\\\\n\\\\nclass Encoder:\\\\n    def __init__(self,batch_size=64):\\\\n        import torch\\\\n        from transformers import AutoTokenizer,AutoModel\\\\n        assert torch.cuda.is_available(),\\\\\'Enable a GPU in Kaggle Settings, then restart/run all.\\\\\'\\\\n        self.torch=torch; self.batch_size=batch_size\\\\n        self.tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False)\\\\n        self.model=AutoModel.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False,\\\\n                                           use_safetensors=True).cuda().half().eval()\\\\n        log(\\\\\'GPU: \\\\\'+torch.cuda.get_device_name(0))\\\\n    def encode(self,frame):\\\\n        texts=text_view(frame,\\\\\'combined\\\\\'); parts=[]; torch=self.torch\\\\n        with torch.inference_mode():\\\\n            for start in range(0,len(texts),self.batch_size):\\\\n                inp=self.tokenizer(texts[start:start+self.batch_size],max_length=256,\\\\n                                   padding=True,truncation=True,return_tensors=\\\\\'pt\\\\\').to(\\\\\'cuda\\\\\')\\\\n                hidden=self.model(**inp).last_hidden_state.float()\\\\n                mask=inp[\\\\\'attention_mask\\\\\'].unsqueeze(-1)\\\\n                vec=(hidden*mask).sum(1)/mask.sum(1).clamp(min=1)\\\\n                vec=torch.nn.functional.normalize(vec,p=2,dim=1)\\\\n                parts.append(vec.cpu().numpy())\\\\n        return np.concatenate(parts).astype(\\\\\'float32\\\\\') if parts else np.empty((0,DIM),np.float32)\\\\n\\\\ndef open_records(work):\\\\n    db=sqlite3.connect(work/\\\\\'records.sqlite\\\\\')\\\\n    db.execute(\\\\\'PRAGMA cache_size=-65536\\\\\')\\\\n    db.execute(\\\\\'CREATE TABLE IF NOT EXISTS records (rid INTEGER PRIMARY KEY, entity_id TEXT, business_name TEXT, business_address TEXT, country TEXT)\\\\\')\\\\n    db.execute(\\\\\'CREATE TABLE IF NOT EXISTS progress (source INTEGER PRIMARY KEY, done INTEGER, complete INTEGER)\\\\\')\\\\n    db.commit()\\\\n    return db\\\\n\\\\ndef encode_pool(data,work,encoder):\\\\n    \\"\\"\\"Vectors flushed before SQLite transaction commits; uncommitted rows overwritten on restart.\\"\\"\\"\\\\n    manifest_path=work/\\\\\'pool_manifest.json\\\\\'\\\\n    if not manifest_path.exists():\\\\n        counts={}\\\\n        for source in (2,3):\\\\n            counts[str(source)]=sum(len(c) for c in read_chunks(data/f\\\\\'train/train_source{source}.tsv\\\\\',[\\\\\'entity_id\\\\\']))\\\\n        atomic_json(manifest_path,{\\\\\'counts\\\\\':counts,\\\\\'total\\\\\':sum(counts.values())})\\\\n    manifest=json.loads(manifest_path.read_text()); total=manifest[\\\\\'total\\\\\']\\\\n    vp=work/\\\\\'vectors.f16\\\\\'; needed=total*DIM*2\\\\n    db=open_records(work)\\\\n    exists=vp.exists()\\\\n    if not exists:\\\\n        assert db.execute(\\\\\'SELECT COUNT(*) FROM records\\\\\').fetchone()[0]==0,\\\\\'Vectors missing but database has progress. Restore both or use new WORK.\\\\\'\\\\n        # Includes space for metadata DB, compressed indexes, model cache and reserve.\\\\n        assert shutil.disk_usage(work).free>needed+5*2**30, \\\\\'Need about 13 GiB free before full-pool encoding. Use a fresh notebook.\\\\\'\\\\n    else: assert vp.stat().st_size==needed,\\\\\'Incomplete vector allocation. Use new WORK directory.\\\\\'\\\\n    vectors=np.memmap(vp,mode=\\\\\'r+\\\\\' if exists else \\\\\'w+\\\\\',dtype=np.float16,shape=(total,DIM))\\\\n    base=0; started=time.time(); encoded=0\\\\n    for source in (2,3):\\\\n        saved=db.execute(\\\\\'SELECT done,complete FROM progress WHERE source=?\\\\\',(source,)).fetchone()\\\\n        done=saved[0] if saved else 0\\\\n        if saved and saved[1]:\\\\n            log(f\\\\\'Source {source}: already encoded {done:,} rows\\\\\'); base+=manifest[\\\\\'counts\\\\\'][str(source)]; continue\\\\n        log(f\\\\\'Source {source}: resume at {done:,}; reading input from beginning to that position\\\\\')\\\\n        seen=0\\\\n        chunks=pd.read_csv(data/f\\\\\'train/train_source{source}.tsv\\\\\',sep=\\\\\'\\\\\\\\t\\\\\',dtype=str,\\\\n                           keep_default_na=False,usecols=COLS,chunksize=10000)\\\\n        for frame in chunks:\\\\n            end=seen+len(frame)\\\\n            if end<=done: seen=end; continue\\\\n            frame=frame.iloc[max(0,done-seen):][COLS]\\\\n            first=max(done,seen); batch=encoder.encode(frame)\\\\n            assert batch.shape==(len(frame),DIM) and np.isfinite(batch).all()\\\\n            vectors[base+first:base+end]=batch.astype(np.float16); vectors.flush()\\\\n            rows=[(base+first+j,str(r.entity_id),str(r.business_name),str(r.business_address),norm(r.country))\\\\n                  for j,r in enumerate(frame.itertuples(index=False))]\\\\n            with db:\\\\n                db.executemany(\\\\\'INSERT INTO records VALUES (?,?,?,?,?)\\\\\',rows)\\\\n                db.execute(\\\\\'INSERT OR REPLACE INTO progress VALUES (?,?,0)\\\\\',(source,end))\\\\n            encoded+=len(frame); seen=end; done=end\\\\n            if end%100000==0:\\\\n                rate=encoded/max(time.time()-started,1)\\\\n                remaining=total-(base+end)\\\\n                log(f\\\\\'Source {source}: {end:,}; {rate:.0f} rows/sec; remaining encoding ~{remaining/rate/60:.1f} min\\\\\')\\\\n                assert shutil.disk_usage(work).free>1.5*2**30,\\\\\'Low disk. Progress saved; free space before resuming.\\\\\'\\\\n        assert done==manifest[\\\\\'counts\\\\\'][str(source)]\\\\n        with db: db.execute(\\\\\'INSERT OR REPLACE INTO progress VALUES (?,?,1)\\\\\',(source,done))\\\\n        base+=done\\\\n    assert db.execute(\\\\\'SELECT COUNT(*) FROM records\\\\\').fetchone()[0]==total\\\\n    atomic_json(work/\\\\\'encoding_complete.json\\\\\',{\\\\\'total\\\\\':total})\\\\n    db.close(); del vectors\\\\n    log(f\\\\\'Encoding complete: {total:,} target records.\\\\\')\\\\n\\\\ndef build_indexes(work):\\\\n    \\"\\"\\"IVF-PQ index per country. Completed countries and trained quantizers are reusable.\\"\\"\\"\\\\n    import faiss\\\\n    faiss.omp_set_num_threads(min(4,os.cpu_count() or 2))\\\\n    total=json.loads((work/\\\\\'encoding_complete.json\\\\\').read_text())[\\\\\'total\\\\\']\\\\n    vectors=np.memmap(work/\\\\\'vectors.f16\\\\\',mode=\\\\\'r\\\\\',dtype=np.float16,shape=(total,DIM))\\\\n    db=open_records(work)\\\\n    countries=[r[0] for r in db.execute(\\\\\'SELECT DISTINCT country FROM records\\\\\')]\\\\n    catalog=[]\\\\n    for country in sorted(countries):\\\\n        token=hashlib.sha256(country.encode()).hexdigest()[:12]\\\\n        path=work/f\\\\\'index_{token}.faiss\\\\\'; trained=work/f\\\\\'trained_{token}.faiss\\\\\'\\\\n        count=db.execute(\\\\\'SELECT COUNT(*) FROM records WHERE country=?\\\\\',(country,)).fetchone()[0]\\\\n        catalog.append({\\\\\'country\\\\\':country,\\\\\'file\\\\\':path.name,\\\\\'count\\\\\':count})\\\\n        if path.exists():\\\\n            check=faiss.read_index(str(path)); assert check.ntotal==count\\\\n            del check; log(f\\\\\'{country}: completed index reused\\\\\'); continue\\\\n        ids=np.fromiter((r[0] for r in db.execute(\\\\\'SELECT rid FROM records WHERE country=? ORDER BY rid\\\\\',(country,))),dtype=np.int64,count=count)\\\\n        # At least 50 training vectors per IVF centroid, up to 100k training vectors.\\\\n        nlist=min(2048,max(1,count//50))\\\\n        if trained.exists(): index=faiss.read_index(str(trained))\\\\n        elif count<10000:\\\\n            index=faiss.IndexIDMap2(faiss.IndexFlatIP(DIM))\\\\n        else:\\\\n            index=faiss.IndexIVFPQ(faiss.IndexFlatIP(DIM),DIM,nlist,48,8,faiss.METRIC_INNER_PRODUCT)\\\\n            index.cp.seed=2026; index.pq.cp.seed=2026\\\\n            train_ids=np.random.default_rng(2026).choice(ids,size=min(count,110000),replace=False)\\\\n            train_vectors=np.asarray(vectors[np.sort(train_ids)],dtype=np.float32)\\\\n            faiss.normalize_L2(train_vectors)\\\\n            log(f\\\\\'{country}: training IVF-PQ on {len(train_vectors):,} vectors; {nlist} cells\\\\\')\\\\n            index.train(train_vectors); del train_vectors\\\\n            faiss.write_index(index,str(trained)+\\\\\'.tmp\\\\\'); os.replace(str(trained)+\\\\\'.tmp\\\\\',trained)\\\\n        started=time.time()\\\\n        # An interrupted country is re-added from its saved trained quantizer.\\\\n        for start in range(0,count,25000):\\\\n            batch_ids=ids[start:start+25000]\\\\n            x=np.asarray(vectors[batch_ids],dtype=np.float32); faiss.normalize_L2(x)\\\\n            index.add_with_ids(x,batch_ids)\\\\n            if start%250000==0: log(f\\\\\'{country}: indexed {min(start+25000,count):,}/{count:,}; {(time.time()-started)/60:.1f} min\\\\\')\\\\n        assert index.ntotal==count\\\\n        faiss.write_index(index,str(path)+\\\\\'.tmp\\\\\'); os.replace(str(path)+\\\\\'.tmp\\\\\',path)\\\\n        del index,ids; gc.collect()\\\\n    atomic_json(work/\\\\\'index_catalog.json\\\\\',catalog)\\\\n    db.close(); del vectors\\\\n    log(\\\\\'All country indexes ready.\\\\\')\\\\n\\\\ndef prepare_queries(data,work,encoder):\\\\n    sample_path=work/\\\\\'sample.pkl\\\\\'\\\\n    if sample_path.exists():\\\\n        with sample_path.open(\\\\\'rb\\\\\') as f: sample,truth=pickle.load(f)\\\\n    else:\\\\n        sample,truth=reproduce_reference_sample(data)\\\\n        with Path(str(sample_path)+\\\\\'.tmp\\\\\').open(\\\\\'wb\\\\\') as f: pickle.dump((sample,truth),f)\\\\n        os.replace(str(sample_path)+\\\\\'.tmp\\\\\',sample_path)\\\\n        sample.to_csv(work/\\\\\'sample.tsv\\\\\',sep=\\\\\'\\\\\\\\t\\\\\',index=False)\\\\n    path=work/\\\\\'query_vectors.npy\\\\\'\\\\n    if path.exists(): q=np.load(path)\\\\n    else:\\\\n        q=encoder.encode(sample)\\\\n        with Path(str(path)+\\\\\'.tmp\\\\\').open(\\\\\'wb\\\\\') as f: np.save(f,q)\\\\n        os.replace(str(path)+\\\\\'.tmp\\\\\',path)\\\\n    assert q.shape==(len(sample),DIM)\\\\n    log(f\\\\\'Sample splits: {sample.split.value_counts().to_dict()}\\\\\')\\\\n    return sample,truth,q\\\\n\\\\ndef retrieve(work,sample,q,nprobe=64,overfetch=200,save_k=50):\\\\n    import faiss\\\\n    query_key=hashlib.sha256((\\\\\'|\\\\\'.join(sample.entity_id)).encode()).hexdigest()[:12]\\\\n    cache=work/f\\\\\'retrieval_{query_key}_p{nprobe}_r{overfetch}_k{save_k}.npz\\\\\'\\\\n    if cache.exists():\\\\n        saved=np.load(cache); return saved[\\\\\'ids\\\\\'],saved[\\\\\'scores\\\\\']\\\\n    total=json.loads((work/\\\\\'encoding_complete.json\\\\\').read_text())[\\\\\'total\\\\\']\\\\n    vectors=np.memmap(work/\\\\\'vectors.f16\\\\\',mode=\\\\\'r\\\\\',dtype=np.float16,shape=(total,DIM))\\\\n    ids=np.full((len(q),save_k),-1,np.int64); scores=np.full((len(q),save_k),-1,np.float32)\\\\n    countries=sample.country.map(norm).to_numpy()\\\\n    for item in json.loads((work/\\\\\'index_catalog.json\\\\\').read_text()):\\\\n        qi=np.flatnonzero(countries==item[\\\\\'country\\\\\'])\\\\n        if not len(qi): continue\\\\n        index=faiss.read_index(str(work/item[\\\\\'file\\\\\']))\\\\n        if hasattr(index,\\\\\'nprobe\\\\\'): index.nprobe=min(nprobe,index.nlist)\\\\n        log(f\\"Searching {len(qi)} businesses in {item[\\\\\'country\\\\\']}: {index.ntotal:,} records\\")\\\\n        for start in range(0,len(qi),64):\\\\n            batch=qi[start:start+64]\\\\n            _,approx=index.search(np.ascontiguousarray(q[batch],dtype=np.float32),overfetch)\\\\n            for i,neighbors in zip(batch,approx):\\\\n                neighbors=np.unique(neighbors[neighbors>=0])\\\\n                if not len(neighbors): continue\\\\n                x=np.asarray(vectors[neighbors],dtype=np.float32)\\\\n                x/=np.maximum(np.linalg.norm(x,axis=1,keepdims=True),1e-12)\\\\n                exact=x@q[i]\\\\n                order=np.lexsort((neighbors,-exact))[:save_k]\\\\n                chosen=neighbors[order]; k=len(chosen)\\\\n                ids[i,:k]=chosen; scores[i,:k]=exact[order]\\\\n        del index; gc.collect()\\\\n    with Path(str(cache)+\\\\\'.tmp\\\\\').open(\\\\\'wb\\\\\') as f: np.savez_compressed(f,ids=ids,scores=scores)\\\\n    os.replace(str(cache)+\\\\\'.tmp\\\\\',cache)\\\\n    return ids,scores\\\\n\\\\ndef fetch_records(work,ids):\\\\n    db=open_records(work); rows={}\\\\n    valid=np.unique(ids[ids>=0])\\\\n    for start in range(0,len(valid),800):\\\\n        batch=[int(x) for x in valid[start:start+800]]\\\\n        for r in db.execute(\\\\\'SELECT * FROM records WHERE rid IN (\\\\\'+\\\\\',\\\\\'.join(\\\\\'?\\\\\'*len(batch))+\\\\\')\\\\\',batch):\\\\n            rows[r[0]]=tuple(r[1:])\\\\n    db.close()\\\\n    assert len(rows)==len(valid)\\\\n    return rows\\\\n\\\\ndef retrieval_report(sample,truth,ids,records):\\\\n    out=[]\\\\n    for split in (\\\\\'train\\\\\',\\\\\'tune\\\\\',\\\\\'evaluation\\\\\'):\\\\n        indices=np.flatnonzero(sample.split.to_numpy()==split)\\\\n        if not len(indices): continue\\\\n        for k in (10,20,50):\\\\n            found=total=0; sizes=[]\\\\n            for i in indices:\\\\n                gt=truth[sample.iloc[i].entity_id]\\\\n                pred={records[int(r)][0] for r in ids[i,:k] if r>=0}\\\\n                found+=len(gt&pred); total+=len(gt); sizes.append(len(pred))\\\\n            out.append(dict(split=split,k=k,found=found,true_links=total,\\\\n                            recall=found/total if total else 1,mean_candidates=float(np.mean(sizes)),\\\\n                            p95_candidates=float(np.percentile(sizes,95)),max_candidates=max(sizes)))\\\\n    return pd.DataFrame(out)\\\\n\\\\nMATCH_FEATURES=FEATURES+[\\\\\'embedding_cosine\\\\\']\\\\ndef make_pairs(sample,truth,ids,scores,records,k=20):\\\\n    x=[]; y=[]; owners=[]; candidate_ids=[]\\\\n    for i,left in enumerate(sample[COLS].itertuples(index=False,name=None)):\\\\n        for rank,rid in enumerate(ids[i,:k]):\\\\n            if rid<0: continue\\\\n            right=records[int(rid)]\\\\n            x.append(features(left,right)+[float(scores[i,rank])])\\\\n            y.append(int(right[0] in truth[left[0]])); owners.append(i); candidate_ids.append(right[0])\\\\n    return np.asarray(x,np.float32),np.asarray(y,np.uint8),np.asarray(owners,np.int32),candidate_ids\\\\n\\\\ndef metrics(sample,truth,owners,candidate_ids,prob,threshold,split):\\\\n    selected=np.flatnonzero(sample.split.to_numpy()==split)\\\\n    pred={int(i):set() for i in selected}\\\\n    for i,c,p in zip(owners,candidate_ids,prob):\\\\n        if int(i) in pred and p>=threshold: pred[int(i)].add(c)\\\\n    scores=[]; tp=fp=fn=singletons=singleton_correct=0\\\\n    for i in selected:\\\\n        gt=truth[sample.iloc[i].entity_id]; pr=pred[int(i)]\\\\n        a=len(gt&pr); b=len(pr-gt); c=len(gt-pr)\\\\n        tp+=a; fp+=b; fn+=c\\\\n        if not gt:\\\\n            singletons+=1; singleton_correct+=int(not pr); scores.append(float(not pr))\\\\n        else: scores.append(1.25*a/(1.25*a+b+.25*c))\\\\n    return dict(macro_F05=float(np.mean(scores)),correct_accepted=tp,incorrect_accepted=fp,\\\\n                missed_true_links=fn,singletons_correct=singleton_correct,singletons=singletons)\\\\n\\\\ndef fit_and_evaluate(work,sample,truth,ids,scores,records):\\\\n    from importlib.metadata import version\\\\n    x,y,owners,candidate_ids=make_pairs(sample,truth,ids,scores,records,k=20)\\\\n    splits=sample.split.to_numpy()[owners]; train=splits==\\\\\'train\\\\\'\\\\n    assert len(np.unique(y[train]))==2,\\\\\'Need positive and negative training pairs.\\\\\'\\\\n    log(f\\\\\'Training new matcher: {train.sum():,} pairs, {y[train].sum():,} true matches\\\\\')\\\\n    model=HistGradientBoostingClassifier(learning_rate=.08,max_iter=250,max_leaf_nodes=31,\\\\n              l2_regularization=1.,early_stopping=False,random_state=2026)\\\\n    with threadpool_limits(limits=4):\\\\n        model.fit(x[train],y[train]); prob=model.predict_proba(x)[:,1]\\\\n    # Only tuning businesses choose the threshold. Evaluation is read once afterwards.\\\\n    tuning=[]\\\\n    for t in np.linspace(.05,.95,91):\\\\n        tuning.append({\\\\\'threshold\\\\\':float(t),**metrics(sample,truth,owners,candidate_ids,prob,t,\\\\\'tune\\\\\')})\\\\n    best=max(tuning,key=lambda d:(d[\\\\\'macro_F05\\\\\'],d[\\\\\'threshold\\\\\']))\\\\n    cutoff=best[\\\\\'threshold\\\\\']\\\\n    evaluation=metrics(sample,truth,owners,candidate_ids,prob,cutoff,\\\\\'evaluation\\\\\')\\\\n    selected=np.flatnonzero(sample.split.to_numpy()==\\\\\'evaluation\\\\\')\\\\n    search_missed=0\\\\n    for i in selected:\\\\n        retrieved={records[int(r)][0] for r in ids[i,:20] if r>=0}\\\\n        search_missed+=len(truth[sample.iloc[i].entity_id]-retrieved)\\\\n    evaluation[\\\\\'missed_by_search\\\\\']=search_missed\\\\n    evaluation[\\\\\'retrieved_but_rejected\\\\\']=evaluation[\\\\\'missed_true_links\\\\\']-search_missed\\\\n    result={\\\\\'version\\\\\':VERSION,\\\\\'model_id\\\\\':MODEL_ID,\\\\\'revision\\\\\':REVISION,\\\\\'candidate_k\\\\\':20,\\\\n            \\\\\'ann_nprobe\\\\\':64,\\\\\'ann_overfetch\\\\\':200,\\\\\'threshold\\\\\':cutoff,\\\\\'tuning\\\\\':best,\\\\n            \\\\\'evaluation\\\\\':evaluation,\\\\\'features\\\\\':MATCH_FEATURES,\\\\n            \\\\\'scope\\\\\':\\\\\'Full training S2/S3; 3000 S1 train, 500 tune, 500 evaluation; no test predictions.\\\\\',\\\\n            \\\\\'versions\\\\\':{p:version(p) for p in [\\\\\'numpy\\\\\',\\\\\'pandas\\\\\',\\\\\'scikit-learn\\\\\',\\\\\'faiss-cpu\\\\\',\\\\\'torch\\\\\',\\\\\'transformers\\\\\']}}\\\\n    atomic_json(work/\\\\\'experiment_results.json\\\\\',result)\\\\n    pd.DataFrame(tuning).to_csv(work/\\\\\'threshold_tuning.csv\\\\\',index=False)\\\\n    with (work/\\\\\'matcher.pkl\\\\\').open(\\\\\'wb\\\\\') as f: pickle.dump({\\\\\'model\\\\\':model,\\\\\'threshold\\\\\':cutoff,\\\\\'features\\\\\':MATCH_FEATURES},f)\\\\n    # Save diagnostics, clearly labeled TRAINING evaluation, not a portal submission.\\\\n    diag=pd.DataFrame({\\\\\'source1_id\\\\\':sample.entity_id.to_numpy()[owners],\\\\\'candidate_id\\\\\':candidate_ids,\\\\n                       \\\\\'split\\\\\':splits,\\\\\'label\\\\\':y,\\\\\'score\\\\\':prob})\\\\n    diag[diag.split!=\\\\\'train\\\\\'].to_csv(work/\\\\\'heldout_pair_scores.tsv\\\\\',sep=\\\\\'\\\\\\\\t\\\\\',index=False)\\\\n    log(json.dumps(result,indent=2))\\\\n    return result\\\\n\'\\n",\n    "(WORK/\'pipeline.py\').write_text(PIPELINE_SOURCE)\\n",\n    "import importlib.util\\n",\n    "spec=importlib.util.spec_from_file_location(\'full_pool_pipeline\', WORK/\'pipeline.py\')\\n",\n    "p=importlib.util.module_from_spec(spec)\\n",\n    "spec.loader.exec_module(p)\\n",\n    "DATA, WORK=p.initialize(DATA,WORK)\\n"\n   ],\n   "execution_count": null,\n   "outputs": [],\n   "id": "cell-04"\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## 1. Encode the full target pool\\n",\n    "This is the main GPU stage. Source 2 and Source 3 are read in batches. Vectors and record text are saved on disk; both must be retained for resume. A completed 10,000-record batch is checkpointed. If GPU memory runs out, reduce BATCH_SIZE to 32 and rerun from the encoder cell. Do not change the model or text format within an existing WORK folder."\n   ],\n   "id": "cell-05"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "encoder=p.Encoder(batch_size=BATCH_SIZE)\\n",\n    "p.encode_pool(DATA,WORK,encoder)\\n",\n    "sample,truth,query_vectors=p.prepare_queries(DATA,WORK,encoder)\\n",\n    "# Release the GPU model after saving the query vectors.\\n",\n    "del encoder\\n",\n    "import gc\\n",\n    "gc.collect()\\n",\n    "torch.cuda.empty_cache()\\n"\n   ],\n   "execution_count": null,\n   "outputs": [],\n   "id": "cell-06"\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## 2. Build the approximate indexes\\n",\n    "This stage uses CPU FAISS. IVF splits vectors into cells; PQ compresses the vectors stored in those cells. Only one country\'s index is built in RAM at a time.\\n",\n    "\\n",\n    "Completed country indexes and trained quantizers are saved. If interrupted while adding vectors, that country\'s additions restart; the expensive embeddings remain reusable. The 200-neighbor search is an intermediate retrieval step. Only the final 20 go to the classifier.\\n"\n   ],\n   "id": "cell-07"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "p.build_indexes(WORK)\\n"\n   ],\n   "execution_count": null,\n   "outputs": [],\n   "id": "cell-08"\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## 3. Measure candidate recall against the full pool\\n",\n    "No ground-truth matches are injected. The labels are consulted only after retrieval to measure success. Recall here is the fraction of true links retrieved, not the challenge macro F0.5 score."\n   ],\n   "id": "cell-09"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "candidate_rows, candidate_scores=p.retrieve(WORK,sample,query_vectors,nprobe=64,overfetch=200,save_k=50)\\n",\n    "records=p.fetch_records(WORK,candidate_rows)\\n",\n    "retrieval=p.retrieval_report(sample,truth,candidate_rows,records)\\n",\n    "retrieval.to_csv(WORK/\'retrieval_summary.csv\',index=False)\\n",\n    "print(retrieval.to_string(index=False))\\n"\n   ],\n   "execution_count": null,\n   "outputs": [],\n   "id": "cell-10"\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## 4. Check whether a more thorough ANN search helps\\n",\n    "On tuning businesses only, double the cells searched and increase intermediate retrieval to 400. This diagnoses approximation losses. It still checks selected cells, not every vector. Similar recall does not prove that exact global nearest neighbors were recovered. The matcher below keeps the predeclared 64-cell / 200-neighbor configuration; this diagnostic informs the next experiment."\n   ],\n   "id": "cell-11"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "import numpy as np\\n",\n    "mask=np.flatnonzero(sample.split.to_numpy()==\'tune\')\\n",\n    "tune_sample=sample.iloc[mask].reset_index(drop=True)\\n",\n    "audit_ids,audit_scores=p.retrieve(WORK,tune_sample,query_vectors[mask],nprobe=128,overfetch=400,save_k=50)\\n",\n    "audit_records=p.fetch_records(WORK,audit_ids)\\n",\n    "audit=p.retrieval_report(tune_sample,truth,audit_ids,audit_records)\\n",\n    "audit[\'search\']=\'128 cells / 400 intermediate neighbors\'\\n",\n    "reference=retrieval[retrieval.split==\'tune\'].copy()\\n",\n    "reference[\'search\']=\'64 cells / 200 intermediate neighbors\'\\n",\n    "import pandas as pd\\n",\n    "ann_audit=pd.concat([reference,audit],ignore_index=True)\\n",\n    "ann_audit.to_csv(WORK/\'ann_tuning_audit.csv\',index=False)\\n",\n    "print(ann_audit.to_string(index=False))\\n",\n    "del audit_records, audit_ids, audit_scores\\n"\n   ],\n   "execution_count": null,\n   "outputs": [],\n   "id": "cell-12"\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## 5. Train a new matching classifier and evaluate\\n",\n    "The classifier learns from 18 lexical comparison features plus embedding cosine similarity. Each of the 20 retrieved pairs receives its match/nonmatch label from the training ground truth. Unretrieved true matches remain misses in end-to-end evaluation.\\n",\n    "\\n",\n    "The classifier is a HistGradientBoostingClassifier. Its acceptance cutoff is chosen using tuning businesses, not evaluation scores. Empty predictions and singleton businesses are included in macro F0.5. No old cutoff or trained matcher is reused.\\n",\n    "\\n",\n    "The 500 evaluation businesses reproduce the earlier baseline split, so this is a useful development comparison. Because that split has been inspected in earlier work, it is not a newly untouched final holdout. A larger fresh holdout will be needed for the next major model selection.\\n"\n   ],\n   "id": "cell-13"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "results=p.fit_and_evaluate(WORK,sample,truth,candidate_rows,candidate_scores,records)\\n",\n    "print(\'\\\\nPASTE BACK THESE RESULTS:\')\\n",\n    "print(retrieval.to_string(index=False))\\n",\n    "print(ann_audit.to_string(index=False))\\n",\n    "print(results[\'evaluation\'])\\n",\n    "print(\'Chosen cutoff:\',results[\'threshold\'])\\n"\n   ],\n   "execution_count": null,\n   "outputs": [],\n   "id": "cell-14"\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## 6. Save the small experiment report\\n",\n    "Download this ZIP after the run completes. It contains metrics, held-out pair scores, the trained matcher, sample rows and runnable pipeline code. It excludes the large vector/database/index caches; those remain in the saved notebook outputs for reuse.\\n",\n    "\\n",\n    "**Do not upload this ZIP to the challenge portal.** There are no test predictions in this experiment. Next, use the full-pool results to decide whether to generate a replacement submission. For that submission, the exported candidate file must contain exactly the candidate sets passed to the matcher, with one row for every test Source 1 entity.\\n"\n   ],\n   "id": "cell-15"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "import json, zipfile\\n",\n    "from importlib.metadata import version\\n",\n    "(WORK/\'requirements_run.txt\').write_text(\'\\\\n\'.join(f\'{name}=={version(name)}\' for name in\\n",\n    "    [\'torch\',\'transformers\',\'numpy\',\'pandas\',\'scikit-learn\',\'faiss-cpu\',\'safetensors\',\'sentencepiece\'])+\'\\\\n\')\\n",\n    "(WORK/\'README.txt\').write_text(\\n",\n    "    \'Full-pool embedding experiment, not a challenge submission.\\\\n\'\\n",\n    "    \'See experiment_results.json, retrieval_summary.csv and ann_tuning_audit.csv.\\\\n\'\\n",\n    "    \'matcher.pkl is generated by this run. Only load trusted pickle files.\\\\n\'\\n",\n    "    \'Full resume requires vectors.f16, records.sqlite, manifests, index files and configuration, \'    \'which are NOT included in this small archive. Retain the saved Kaggle version outputs.\\\\n\')\\n",\n    "files=[\'configuration.json\',\'pool_manifest.json\',\'experiment_results.json\',\'retrieval_summary.csv\',\\n",\n    "       \'ann_tuning_audit.csv\',\'threshold_tuning.csv\',\'heldout_pair_scores.tsv\',\'matcher.pkl\',\\n",\n    "       \'sample.tsv\',\'pipeline.py\',\'requirements_run.txt\',\'README.txt\']\\n",\n    "archive=Path(\'/kaggle/working/full_pool_embedding_results.zip\')\\n",\n    "with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:\\n",\n    "    for name in files:\\n",\n    "        assert (WORK/name).exists(), f\'Missing result: {name}\'\\n",\n    "        z.write(WORK/name,arcname=name)\\n",\n    "print(\'Experiment complete. Download:\',archive)\\n",\n    "from IPython.display import FileLink,display\\n",\n    "display(FileLink(str(archive)))\\n"\n   ],\n   "execution_count": null,\n   "outputs": [],\n   "id": "cell-16"\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "### Interpreting the result\\n",\n    "- Compare full-pool **top-20 recall** with the old exact-key recall (~0.696 on its 500-business evaluation). The pilot\'s 0.969 recall came from a smaller pool and is not an expectation for this run.\\n",\n    "- Compare development **macro F0.5** with the old baseline\'s ~0.788. Your **0.756 leaderboard score** is on different, hidden data.\\n",\n    "- `missed_by_search` measures retrieval failures; `retrieved_but_rejected` measures true candidates the matcher discarded.\\n",\n    "- A low incorrect-acceptance count matters, but rejecting too much can also reduce the score.\\n",\n    "- France appears in test but not training. These training results do not establish French performance.\\n",\n    "\\n",\n    "Reference implementations: [E5 model card](https://huggingface.co/intfloat/multilingual-e5-small) and [FAISS index documentation](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes). No challenge business data is sourced from these links.\\n"\n   ],\n   "id": "cell-17"\n  }\n ],\n "metadata": {\n  "kernelspec": {\n   "display_name": "Python 3",\n   "language": "python",\n   "name": "python3"\n  },\n  "language_info": {\n   "name": "python",\n   "version": "3.12"\n  },\n  "kaggle": {\n   "accelerator": "gpu",\n   "isInternetEnabled": true,\n   "language": "python",\n   "sourceType": "notebook"\n  }\n },\n "nbformat": 4,\n "nbformat_minor": 5\n}'
(WORK/'embedding_core.py').write_text(CORE_SOURCE)
(WORK/'submit_pipeline.py').write_text(RUNNER_SOURCE)
(WORK/'training_experiment.ipynb').write_text(TRAINING_NOTEBOOK)
# Use a subprocess for dependency isolation and to release memory between stages.
import subprocess,json
probe=subprocess.run([sys.executable,'-c',
    "import submit_pipeline as s; print(s.locate_model("+repr(MODEL_DIR or None)+"))"],
    cwd=WORK,text=True,capture_output=True)
if probe.returncode:
    print(probe.stderr)
    raise RuntimeError('Attach the saved experiment output, or set MODEL_DIR to its model folder.')
MODEL_DIR=Path(probe.stdout.strip())
meta=json.loads((MODEL_DIR/'experiment_results.json').read_text())
print('Using trained model:',MODEL_DIR/'matcher.pkl')
print('Previous local evaluation:',meta['evaluation']['macro_F05'])
print('Cutoff:',meta['threshold'])
assert meta['candidate_k']==20 and meta['ann_nprobe']==64 and meta['ann_overfetch']==200

def run_stage(stage):
    cmd=[sys.executable,str(WORK/'submit_pipeline.py'),'--data',str(DATA),
         '--work',str(WORK),'--model-dir',str(MODEL_DIR),'--stage',stage,
         '--batch-size',str(BATCH_SIZE)]
    with (WORK/f'{stage}.log').open('a') as log:
        process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line);log.flush()
        rc=process.wait()
    if rc:raise RuntimeError(f'{stage} failed. Check the log above. Completed checkpoints remain in WORK.')


## 1. Encode test Source 2/3
All test target records are encoded. Every 10,000 completed records are checkpointed. If a GPU memory error occurs, reduce BATCH_SIZE to 32 and rerun this stage. Encoding does not fine-tune the E5 model.

Progress can resume only when WORK and its caches still exist. A new Save & Run All session may start with empty working storage; do not assume it inherits an unsaved interactive run.

In [ ]:
run_stage('encode')


## 2. Index the test pool
Country-specific IVF-PQ indexes are created dynamically, including France. Index training is an unlabeled compression/search step; the saved matching classifier is unchanged.
Completed country indexes are reusable. An interrupted country's additions restart from its saved trained quantizer.

In [ ]:
run_stage('index')


## 3. Predict all test businesses
For each Source 1 business, retrieve up to 200 approximate neighbors, rerank them by cosine similarity, and keep at most **20**. Score those 20 with the saved matcher and apply its saved cutoff (0.70 in your run).

The candidate file records **all pairs scored by the classifier**, including rejected pairs. Both files preserve one row per Source 1 business, including empty results. Prediction shards are saved every 2,000 businesses, allowing a same-filesystem rerun to skip completed batches.

In [ ]:
run_stage('predict')


## 4. Merge, validate, and package
This runs the organizer's validator with `--check-ids`, so IDs are checked against the test dataset too. If validation fails, the cell stops; do not upload an unvalidated file.

Only `matching_results.tsv` goes to the leaderboard portal. Keep `candidate_pairs.tsv` and the backup for final review. The backup includes runnable prediction code, the training notebook, model, dependency versions, methodology and validation log. At final closure, complete the organizer's documentation template and any additional packaging instructions.

In [ ]:
run_stage('finalize')


## Download your submission
If a TSV link opens a preview, download **matching_results_download.zip**, extract it on your computer, and upload the extracted **matching_results.tsv** to the challenge portal. Do not upload the download ZIP as the leaderboard prediction file.

The backup ZIP excludes large vector/index caches. Keep the completed saved Kaggle version if you want to reuse those caches later.

In [ ]:
from IPython.display import FileLink,display
print('PORTAL: upload this TSV after validator PASS')
display(FileLink(str(WORK/'output/matching_results.tsv')))
print('EASIER DOWNLOAD: extract this ZIP to obtain that same TSV')
display(FileLink(str(WORK/'output/matching_results_download.zip')))
print('KEEP FOR FINAL REVIEW:')
display(FileLink(str(WORK/'output/candidate_pairs.tsv')))
display(FileLink(str(WORK.parent/'embedding_submission_v1_backup.zip')))
print(json.dumps(json.loads((WORK/'output/submission_summary.json').read_text()),indent=2))
